In [ ]:
%run ./phase2_common

In [ ]:
import json

dbutils.widgets.text("input_path", "", "Raw JSON file or folder to check")
input_path = dbutils.widgets.get("input_path").strip()
if not input_path:
    raise ValueError("input_path is required")

files = (
    spark.read.option("multiLine", True)
    .schema("load_type STRING, record_count BIGINT, records ARRAY<STRING>")
    .json(input_path)
)
assert files.filter(F.col("record_count") != F.size("records")).count() == 0, "record_count mismatch"

first = json.loads(files.select(F.explode("records")).first()[0])
for field in ("id", "title", "genres", "production_companies", "credits", "keywords", "reviews"):
    assert field in first, f"Missing field in raw payload: {field}"
print("Raw file contract OK")

In [ ]:
def parse(rows):
    df = spark.createDataFrame(rows, "test_case STRING, bronze_movie_id BIGINT, raw_payload STRING")
    return (
        df.withColumn("movie", F.from_json("raw_payload", MOVIE_SCHEMA))
        .select(
            "test_case", "bronze_movie_id",
            F.col("movie.id").alias("id"), F.col("movie.title").alias("title"),
            F.lit(f"contract-test-{uuid4()}").alias("batch_id"),
            F.lit("synthetic").alias("source_file"),
            F.current_timestamp().alias("source_extracted_at"),
        )
    )


cases = parse([
    ("additive-field", 900000001, '{"id":900000001,"title":"ok","new_source_field":"kept in Bronze only"}'),
    ("wrong-type", 900000002, '{"id":"not-a-number","title":"bad"}'),
    ("blank-title", 900000003, '{"id":900000003,"title":"  "}'),
])
checked = with_validation_reason(cases)
batch = checked.select("batch_id").first()[0]
invalid = checked.filter(F.col("validation_reason").isNotNull())

assert checked.filter(F.col("validation_reason").isNull()).count() == 1, "additive field must be accepted"
assert invalid.count() == 2, "incompatible records must be rejected"

try:
    merge_quarantine(invalid)
    merge_quarantine(invalid)
    stored = spark.table(QUARANTINE).filter(F.col("batch_id") == batch).count()
    assert stored == 2, f"Quarantine must be idempotent; found {stored} rows"
finally:
    spark.sql(f"DELETE FROM {QUARANTINE} WHERE batch_id = '{batch}'")
print("Schema drift and quarantine tests OK")